# Module 1: Linear Regression and KMeans Clustering

**Objective:** Understand two foundational ML algorithms -- one supervised (Linear Regression) and one unsupervised (KMeans Clustering). We will build intuition from theory, then implement each from scratch and with scikit-learn.

---

## Part A: Linear Regression

### 1.1 What Problem Does Linear Regression Solve?

Imagine you have data about houses: the area in square feet and the sale price. You notice that bigger houses tend to cost more. Linear regression finds the **best-fitting straight line** through your data so you can predict the price of a new house given its area.

Mathematically, we model the relationship as:

$$y = w \cdot x + b$$

Where:
- **y** is the target (price)
- **x** is the feature (area)
- **w** is the weight (slope) -- how much y changes per unit change in x
- **b** is the bias (intercept) -- the baseline value of y when x = 0

### 1.2 How Does It Learn? (The Cost Function)

The model starts with random w and b. For each data point, it makes a prediction and measures the error. The **Mean Squared Error (MSE)** aggregates all errors:

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

The algorithm adjusts w and b to **minimize** this cost. This process is called **Gradient Descent** -- we compute the gradient (slope of the cost surface) and take small steps downhill.

### 1.3 Gradient Descent -- Step by Step

1. Initialize w and b to small random values (or zero).
2. Compute predictions: y_hat = w * x + b
3. Compute gradients:
   - dw = (-2/n) * sum(x_i * (y_i - y_hat_i))
   - db = (-2/n) * sum(y_i - y_hat_i)
4. Update parameters:
   - w = w - learning_rate * dw
   - b = b - learning_rate * db
5. Repeat until convergence.

The **learning rate** controls step size. Too large and you overshoot; too small and training is slow.

### 1.4 Let's Implement It From Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Generate synthetic data ---
# Scenario: Predicting house price (in $1000s) from area (in 100 sq ft units)
np.random.seed(42)
X = 2 * np.random.rand(100, 1)          # area: 0 to 200 (in 100 sqft)
y = 4 + 3 * X + np.random.randn(100, 1)  # true relationship: y = 4 + 3x + noise

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.5)
plt.xlabel('Area (x100 sq ft)')
plt.ylabel('Price ($1000s)')
plt.title('House Price vs Area -- Raw Data')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Linear Regression from scratch ---

class LinearRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.w = None
        self.b = None
        self.cost_history = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.w = np.zeros((X.shape[1], 1))
        self.b = 0.0

        for i in range(self.n_iter):
            # Forward pass: compute predictions
            y_hat = X @ self.w + self.b

            # Compute cost (MSE)
            cost = np.mean((y - y_hat) ** 2)
            self.cost_history.append(cost)

            # Compute gradients
            dw = (-2 / n_samples) * (X.T @ (y - y_hat))
            db = (-2 / n_samples) * np.sum(y - y_hat)

            # Update parameters
            self.w -= self.lr * dw
            self.b -= self.lr * db

        return self

    def predict(self, X):
        return X @ self.w + self.b

# Train the model
model = LinearRegressionScratch(learning_rate=0.1, n_iterations=1000)
model.fit(X, y)

print(f'Learned weight (w): {model.w[0][0]:.4f}  (true: 3.0)')
print(f'Learned bias (b):   {model.b:.4f}  (true: 4.0)')

In [ ]:
# --- Visualize the cost function decreasing ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cost history
axes[0].plot(model.cost_history)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE Cost')
axes[0].set_title('Cost Function Over Training')
axes[0].grid(True, alpha=0.3)

# Plot 2: Fitted line
axes[1].scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.5, label='Data')
X_line = np.linspace(0, 2, 100).reshape(-1, 1)
y_line = model.predict(X_line)
axes[1].plot(X_line, y_line, 'r-', linewidth=2, label='Fitted Line')
axes[1].set_xlabel('Area (x100 sq ft)')
axes[1].set_ylabel('Price ($1000s)')
axes[1].set_title('Linear Regression -- Fitted Line')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.5 Using scikit-learn (Production Approach)

In practice, you use well-tested libraries. scikit-learn uses the **Normal Equation** (closed-form solution) rather than iterative gradient descent for linear regression, which is faster for small datasets.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
sk_model = LinearRegression()
sk_model.fit(X_train, y_train)

# Evaluate
y_pred = sk_model.predict(X_test)
print(f'sklearn weight: {sk_model.coef_[0][0]:.4f}')
print(f'sklearn bias:   {sk_model.intercept_[0]:.4f}')
print(f'MSE on test set: {mean_squared_error(y_test, y_pred):.4f}')
print(f'R-squared:       {r2_score(y_test, y_pred):.4f}')
print()
print('R-squared interpretation: 1.0 = perfect prediction, 0.0 = model is no better than predicting the mean.')

### 1.6 Key Takeaways -- Linear Regression

- Linear regression models a **linear** relationship between features and target.
- It learns by **minimizing MSE** using gradient descent (or closed-form solution).
- The **learning rate** is a critical hyperparameter.
- **R-squared** tells you how much variance your model explains.
- It assumes: linearity, no multicollinearity among features, homoscedasticity (constant variance of errors).

---

## Part B: KMeans Clustering

### 1.7 What Problem Does KMeans Solve?

Unlike linear regression, KMeans is **unsupervised** -- there are no labels. Given a set of data points, KMeans groups them into **K clusters** such that points in the same cluster are more similar to each other than to points in other clusters.

Real-world uses: customer segmentation, document grouping, image compression, anomaly detection.

### 1.8 The Algorithm

1. **Choose K** (the number of clusters).
2. **Initialize** K centroids randomly (pick K random data points).
3. **Assign** each data point to the nearest centroid (using Euclidean distance).
4. **Update** each centroid to be the mean of all points assigned to it.
5. **Repeat** steps 3-4 until centroids stop moving (convergence).

The objective function (inertia) that KMeans minimizes:

$$J = \sum_{k=1}^{K}\sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

Where mu_k is the centroid of cluster C_k.

### 1.9 KMeans From Scratch

In [ ]:
# --- Generate synthetic clustered data ---
from sklearn.datasets import make_blobs

X_cluster, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(X_cluster[:, 0], X_cluster[:, 1], alpha=0.6, edgecolors='k', linewidth=0.5)
plt.title('Unlabeled Data -- Can You See the Clusters?')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
class KMeansScratch:
    def __init__(self, k=3, max_iter=100, random_state=42):
        self.k = k
        self.max_iter = max_iter
        self.rng = np.random.RandomState(random_state)
        self.centroids = None
        self.labels = None
        self.inertia_history = []

    def fit(self, X):
        n_samples = X.shape[0]

        # Step 1: Initialize centroids by picking K random data points
        idx = self.rng.choice(n_samples, self.k, replace=False)
        self.centroids = X[idx].copy()

        for iteration in range(self.max_iter):
            # Step 2: Assign each point to nearest centroid
            distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
            self.labels = np.argmin(distances, axis=1)

            # Compute inertia
            inertia = sum(np.sum((X[self.labels == k] - self.centroids[k]) ** 2)
                          for k in range(self.k))
            self.inertia_history.append(inertia)

            # Step 3: Update centroids
            new_centroids = np.array([X[self.labels == k].mean(axis=0)
                                      for k in range(self.k)])

            # Check for convergence
            if np.allclose(self.centroids, new_centroids):
                print(f'Converged at iteration {iteration + 1}')
                break
            self.centroids = new_centroids

        return self

# Fit
kmeans = KMeansScratch(k=4)
kmeans.fit(X_cluster)

# Visualize
plt.figure(figsize=(8, 5))
plt.scatter(X_cluster[:, 0], X_cluster[:, 1], c=kmeans.labels, cmap='viridis',
            alpha=0.6, edgecolors='k', linewidth=0.5)
plt.scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1],
            c='red', marker='X', s=200, edgecolors='k', linewidth=1.5, label='Centroids')
plt.title('KMeans Clustering -- From Scratch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 1.10 How to Choose K -- The Elbow Method

One fundamental question: how many clusters? The **Elbow Method** runs KMeans for different values of K and plots the inertia. The "elbow" point (where adding more clusters gives diminishing returns) suggests a good K.

In [ ]:
from sklearn.cluster import KMeans

inertias = []
K_range = range(1, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.title('Elbow Method for Optimal K')
plt.grid(True, alpha=0.3)
plt.show()
print('The elbow appears at K=4, matching our true number of clusters.')

### 1.11 scikit-learn KMeans

In [ ]:
sk_km = KMeans(n_clusters=4, random_state=42, n_init=10)
sk_labels = sk_km.fit_predict(X_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_cluster[:, 0], X_cluster[:, 1], c=y_true, cmap='viridis',
                alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].set_title('True Labels (Ground Truth)')

axes[1].scatter(X_cluster[:, 0], X_cluster[:, 1], c=sk_labels, cmap='viridis',
                alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].scatter(sk_km.cluster_centers_[:, 0], sk_km.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='k', linewidth=1.5)
axes[1].set_title('sklearn KMeans Predictions')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1.12 Key Takeaways -- KMeans

- KMeans is **unsupervised**: no labels needed.
- It partitions data into K clusters by minimizing within-cluster variance.
- You must **choose K** beforehand (use the Elbow Method, Silhouette Score, etc.).
- It assumes clusters are **spherical and equally sized** (a limitation).
- Initialization matters -- scikit-learn runs it multiple times (n_init) to avoid bad starts.
- **Contrast with Linear Regression**: regression predicts a continuous value; clustering discovers structure.

---

### 1.13 Connecting the Concepts

| Aspect | Linear Regression | KMeans Clustering |
|---|---|---|
| Type | Supervised | Unsupervised |
| Goal | Predict a continuous target | Discover groups in data |
| Learns from | (X, y) pairs | X only |
| Optimization | Minimize MSE | Minimize inertia |
| Output | A prediction (number) | A cluster label (integer) |
| Key hyperparameter | Learning rate | K (number of clusters) |